In [38]:
## nba_api try

## Notes it only works from 1983 on.

from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd

In [39]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time

def get_all_nba_schedules(start_year=1983, end_year=2025):
    """
    Get all NBA regular season schedules from start_year to end_year.
    
    Parameters:
    - start_year: First year to fetch (default: 1983)
    - end_year: Last year to fetch (default: 2025)
    
    Returns:
    - DataFrame with all games
    """
    all_games = []
    
    for year in range(start_year, end_year):
        # NBA seasons are formatted as 'YYYY-YY' (e.g., '1983-84')
        season = f"{year}-{str(year + 1)[-2:]}"
        
        print(f"Fetching season {season}...")
        
        try:
            # Get games for this season
            gamefinder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00',  # '00' = NBA
                season_type_nullable='Regular Season'
            )
            
            games = gamefinder.get_data_frames()[0]
            all_games.append(games)
            
            print(f"  Found {len(games)} game records for {season}")
            
            # Be respectful to the API - add a small delay
            time.sleep(0.6)
            
        except Exception as e:
            print(f"  Error fetching {season}: {e}")
            continue
    
    # Combine all seasons into one DataFrame
    if all_games:
        full_schedule = pd.concat(all_games, ignore_index=True)
        print(f"\nTotal game records: {len(full_schedule)}")
        return full_schedule
    else:
        print("No games found")
        return pd.DataFrame()

In [ ]:
schedules = get_all_nba_schedules(1983, 2025)

In [ ]:
all_teams = set(schedules['TEAM_ABBREVIATION'])

season_ids = set(schedules['SEASON_ID'])

franchise_map = {
    # Utah Jazz
    "UTA": "Utah Jazz",
    "UTH": "Utah Jazz",

    # Golden State Warriors
    "GSW": "Golden State Warriors",
    "GOS": "Golden State Warriors",

    # San Antonio Spurs
    "SAS": "San Antonio Spurs",
    "SAN": "San Antonio Spurs",

    # Philadelphia 76ers
    "PHI": "Philadelphia 76ers",
    "PHL": "Philadelphia 76ers",

    # Sacramento / Kansas City Kings
    "SAC": "Sacramento Kings",
    "KCK": "Sacramento Kings",

    # Vancouver / Memphis Grizzlies
    "VAN": "Memphis Grizzlies",
    "MEM": "Memphis Grizzlies",

    # New Jersey / Brooklyn Nets
    "NJN": "Brooklyn Nets",
    "BKN": "Brooklyn Nets",

    # San Diego / LA Clippers
    "SDC": "Los Angeles Clippers",
    "LAC": "Los Angeles Clippers",

    # New Orleans franchise (Hornets → Pelicans)
    "NOH": "New Orleans Pelicans",
    "NOK": "New Orleans Pelicans",
    "NOP": "New Orleans Pelicans",

    # Charlotte franchise (Bobcats → Hornets)
    "CHH": "Charlotte Hornets",
    "CHA": "Charlotte Hornets",

    # Single-abbreviation franchises
    "LAL": "Los Angeles Lakers",
    "BOS": "Boston Celtics",
    "CHI": "Chicago Bulls",
    "NYK": "New York Knicks",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "CLE": "Cleveland Cavaliers",
    "DET": "Detroit Pistons",
    "IND": "Indiana Pacers",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "ATL": "Atlanta Hawks",
    "MIA": "Miami Heat",
    "ORL": "Orlando Magic",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "TOR": "Toronto Raptors",
    "OKC": "Oklahoma City Thunder",
    "SEA": "Oklahoma City Thunder",  # SuperSonics history stays with Thunder
    "WAS": "Washington Wizards",
    "HOU": "Houston Rockets"
}

In [ ]:
def generateDataFrame(schedules, season_ids, franchise_map):
    """
    Build a DataFrame where each row corresponds to a (season, franchise)
    pair and contains the full win/loss sequence for that team-season.

    Parameters
    ----------
    schedules : pd.DataFrame
        Game-level schedule data containing at least:
        - TEAM_ABBREVIATION
        - SEASON_ID
        - WL (win/loss indicator)

    season_ids : iterable
        Collection of season identifiers (e.g. 22017, 22018, ...)

    franchise_map : dict
        Mapping from team abbreviations to franchise identifiers

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - League
        - Season
        - Team (franchise)
        - Sequence (list of binary win/loss outcomes)
    """

    rows = []

    for season in season_ids:
        for team in all_teams:
            temp = schedules[(schedules['TEAM_ABBREVIATION'] == team) & (schedules['SEASON_ID'] == season)]
            if temp.empty:
                continue

            wl = list(temp["WL"])
            binary_sequence = [1 if x == "W" else 0 for x in wl]

            rev_binary_sequence = binary_sequence[::-1] ## Noticed that Sequence Data was reversed. Reversing it back.

            rows.append({
                "League" : "NBA",
                "Season" : str(int(season) - 20000), ## Conversion from schedule season id. 
                "Team" : franchise_map[team],
                "Sequence" : rev_binary_sequence,
            })

    out = pd.DataFrame(rows)

    return out

In [43]:
# Merge missing NBA Data to us_leagues.csv

us_leagues = pd.read_csv("../data/processed/us_leagues.csv")

In [ ]:
## Writing out to processed data directory

output = "../data/raw/nba.csv"

main = generateDataFrame(schedules, season_ids, franchise_map)

main.to_csv(output)

In [ ]:
# Checks for incomplete data.

df = pd.read_csv("../data/raw/nba.csv")

df["Length"] = len(df['Sequence'])

df[df["Length"] != 1163] 

import ast

a = df.iloc[842].Sequence

lst = ast.literal_eval(a)

df['Sequence_List'] = df["Sequence"].apply(ast.literal_eval)

df['Sequence_List_Length'] = df["Sequence_List"].apply(len)

df[df["Sequence_List_Length"] != 82]

## Note these seasons are okay. Some of these were due to lockouts, COVID, or other cancellation reasons.

In [44]:
## Check Missing Years

years = set(us_leagues[us_leagues['League'] == "NBA"]['Season'])

for i in range(1983, 2025):
    if i not in years:
        print(i)

In [45]:
## Append Missing Years

df_2015 = df[df['Season'] == 2015]

us_leagues_filled_missing = pd.concat([df_2015,us_leagues], ignore_index=True)

In [46]:
## Check Again for Missing Years

years = set(us_leagues_filled_missing[us_leagues_filled_missing['League'] == "NBA"]['Season'])

for i in range(1983, 2025):
    if i not in years:
        print(i)

In [ ]:
## Rewrite us_leagues.csv

output = "../data/processed/us_leagues.csv"

us_leagues_filled_missing.to_csv(output)

In [190]:
# Implement Later for More Accurate Data Collection

import time, random
import pandas as pd
year = 2016

def get_year(year) -> list:

    print(f'{year} season')

    months = ["october", "november", "december", "january", "february", "march", "april"]
    missing = []

    for month in months:
        url = f'https://www.basketball-reference.com/leagues/NBA_{year}_games-{month}.html'
        out_path = f"../data/raw/rescrape_nba/NBA_{month}-{year}.csv"

        print(f'    Fetching {month}...')

        try: 
            tables = pd.read_html(url)
            df = tables[0]

            ## Remove Playoff Rows
            if (df['Date'] == "Playoffs").any():                
                idx = df[df['Date'] == "Playoffs"].index[0]     
                df = df.iloc[:idx,]                             

            df.to_csv(out_path, index = False)

            print(f'    {year} {month} success!')
            
        except Exception as e:
            print(f'Error fetching {month} {year}')
            print(e)
            missing.append( ({year}, {month}) )

        time.sleep(random.uniform(4, 5))

    return missing

In [ ]:
years = list(range(1955,1983))
for year in years:

    get_year(year)

1955 season
    Fetching october...
    1955 october success!
    Fetching november...
    1955 november success!
    Fetching december...
    1955 december success!
    Fetching january...
    1955 january success!
    Fetching february...
    1955 february success!
    Fetching march...
    1955 march success!
    Fetching april...
    1955 april success!
1956 season
    Fetching october...
Error fetching october 1956
HTTP Error 404: Not Found
    Fetching november...
    1956 november success!
    Fetching december...
    1956 december success!
    Fetching january...
    1956 january success!
    Fetching february...
    1956 february success!
    Fetching march...
    1956 march success!
    Fetching april...
    1956 april success!
1957 season
    Fetching october...
    1957 october success!
    Fetching november...
    1957 november success!
    Fetching december...
    1957 december success!
    Fetching january...
    1957 january success!
    Fetching february...
    1957 fe

In [192]:
## Getting 1982-83 Season.

get_year(1983)

1983 season
    Fetching october...
    1983 october success!
    Fetching november...
    1983 november success!
    Fetching december...
    1983 december success!
    Fetching january...
    1983 january success!
    Fetching february...
    1983 february success!
    Fetching march...
    1983 march success!
    Fetching april...
    1983 april success!


[]

In [196]:
import os
import pandas as pd
from collections import defaultdict

def load_yearly_data(directory):

    yearly_data = defaultdict(list)

    for file in os.listdir(directory):

        if file.endswith(".csv"):

            # extract year from filename
            year = file.split("-")[1].replace(".csv", "")

            path = os.path.join(directory, file)

            df = pd.read_csv(path)

            # convert date column to datetime
            df["date"] = pd.to_datetime(df["Date"], format="%a, %b %d, %Y")

            yearly_data[year].append(df)

    # combine each year's dataframes
    yearly_dfs = {}

    for year, dfs in yearly_data.items():

        combined = pd.concat(dfs, ignore_index=True)

        # sort chronologically
        combined = combined.sort_values("date")

        yearly_dfs[year] = combined

    return yearly_dfs

In [197]:
yearly = load_yearly_data("../data/raw/rescrape_nba")

In [228]:
def extract(row) -> tuple:

    t1 = row['Visitor/Neutral']
    t2 = row['Home/Neutral']
    t1_score = row['PTS']
    t2_score = row['PTS.1']

    if t1_score > t2_score:
        W, L = t1, t2
    else:
        W, L = t2, t1

    return (W, L)

def get_year_sequences(yr) -> dict:

    sequences = defaultdict(list)

    yr = yr.apply(lambda row: extract(row), axis = 1)

    for winner, loser in yr:
        sequences[winner].append(1)
        sequences[loser].append(0)

    return sequences

def format_year(seq, season) -> pd.DataFrame:

    output = []
    for team, sequence in seq.items():
        output.append({
            'League': "NBA",
            'Season': season,
            'Team': team,
            'Sequence': sequence
        })
    return pd.DataFrame(output)

In [244]:
all_years = []

for season, year_df in yearly.items():  # yearly is your dict of dfs per season

    # Build binary win/loss sequences per team
    sequences = get_year_sequences(year_df)
    
    # Format this season as a DataFrame and add to the list
    season_df = format_year(sequences, season)
    all_years.append(season_df)

final_df = pd.concat(all_years, ignore_index=True)

In [245]:
## Change Season Convention to Match NBA Dataset.

final_df['Season'] = final_df['Season'].apply(int) - 1

us_leagues = pd.read_csv('../data/processed/us_leagues.csv')

us_leagues = us_leagues[['League', 'Season', 'Team', 'Sequence']]

In [246]:
## Add 1955-1982

updated = pd.concat([us_leagues, final_df], ignore_index=True)

updated.to_csv('../data/processed/us_leagues.csv', index = False)